In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pymupdf
import qdrant_client
import pydantic
import langchain
import re

In [3]:
from typing import List, Tuple, Dict, Any
from uuid import uuid4
from pydantic import BaseModel, Field, computed_field

class ChunkMetadata(BaseModel):
    chunking_method: str = Field(..., description="Method used for chunking the document")
    source_info: str = Field(..., description="Source information for the document chunk")
    # Add optional strict fields here, or allow open dicts
    extra_details: Dict[str, Any] = Field(default_factory=dict)

class DocumentChunk(BaseModel):
    id: str = Field(...)
    source: str = Field(..., description="Source of the document chunk")
    chunk_index : int = Field(..., description="Index of the chunk within the document")
    content: str = Field(..., description="Content of the document chunk")
    metadata: ChunkMetadata = Field(default_factory=ChunkMetadata, description="Additional metadata for the document chunk")
    @computed_field
    def length(self) -> int:
        return len(self.content)

class ChunkCollection(BaseModel):
    chunks: List[DocumentChunk] = Field(default_factory=list, description="List of document chunks in the collection")





In [4]:

def extract_text_from_pdf(pdf_path, source,):
    doc = pymupdf.open(pdf_path) 
    pages_data = [ 
     # iterate the document pages
       {
           "source": source,
           "page_number": index,
           "text" : page.get_text(),
           "num_of_pages": doc.page_count
       }
       for index, page in enumerate(doc)
    ]
    doc.close()
    return pages_data
    

In [5]:
texts = extract_text_from_pdf("../eu_ai_act.pdf", "eu_ai_act")


In [6]:
print(texts[3])
print(texts[3]["num_of_pages"])

{'source': 'eu_ai_act', 'page_number': 3, 'text': 'rights and guarantees awarded to them by such Union law, including the rights related to solely automated individual \ndecision-making, including profiling. Harmonised rules for the placing on the market, the putting into service and \nthe use of AI systems established under this Regulation should facilitate the effective implementation and enable the \nexercise of the data subjects’ rights and other remedies guaranteed under Union law on the protection of personal \ndata and of other fundamental rights.\n(11)\nThis Regulation should be without prejudice to the provisions regarding the liability of providers of intermediary \nservices as set out in Regulation (EU) 2022/2065 of the European Parliament and of the Council (15).\n(12)\nThe notion of ‘AI system’ in this Regulation should be clearly defined and should be closely aligned with the work of \ninternational organisations working on AI to ensure legal certainty, facilitate interna

In [7]:
def clean_text(text: str) -> str:
    """Cleans text safely by removing non-printables and extra whitespace while preserving paragraph structure."""
    text = text.replace('\\n', ' ')
    text = text.replace('\\t', ' ')
    text = text.replace('\\r', ' ')
    text = re.sub(r'\\[a-zA-Z]', ' ', text)
    # 1. Fix broken hyphenation across line breaks (e.g., "bro-\nken")
    text = re.sub(r"-\s*\n\s*", "", text)
 
    
    # 2. Filter non-printable characters (keep tabs and actual newlines)
    text = "".join(c for c in text if c.isprintable() or c in ['\n', '\t'])
    
    # 3. Normalize extra spaces/tabs on a single line
    text = re.sub(r'[ \t]+', ' ', text)
    
    # 4. Strip trailing spaces per line and remove completely blank lines
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    # 5. Join clean lines back together
    return '\n'.join(lines)

In [8]:
clean_texts = [ { **text, "text": clean_text(text["text"]) } for text in texts ]
print(clean_texts[3])
print(clean_texts[3]["num_of_pages"])
clean_texts_length = sum(len(text["text"]) for text in clean_texts)
print(clean_texts_length)
cleaned = "\n ---Page Break---\n".join(text["text"] for text in clean_texts)
print(cleaned)
len(cleaned)

{'source': 'eu_ai_act', 'page_number': 3, 'text': 'rights and guarantees awarded to them by such Union law, including the rights related to solely automated individual\ndecision-making, including profiling. Harmonised rules for the placing on the market, the putting into service and\nthe use of AI systems established under this Regulation should facilitate the effective implementation and enable the\nexercise of the data subjects’ rights and other remedies guaranteed under Union law on the protection of personal\ndata and of other fundamental rights.\n(11)\nThis Regulation should be without prejudice to the provisions regarding the liability of providers of intermediary\nservices as set out in Regulation (EU) 2022/2065 of the European Parliament and of the Council (15).\n(12)\nThe notion of ‘AI system’ in this Regulation should be clearly defined and should be closely aligned with the work of\ninternational organisations working on AI to ensure legal certainty, facilitate international

598279

In [9]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
# Import HuggingFaceEmbeddings from langchain_community.embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
chunker = SemanticChunker(embeddings=embeddings)
chunks = chunker.create_documents([cleaned])
chunks[0]


/var/folders/74/60f721fx35g_3fc4z0gcl2gm0000gn/T/ipykernel_46283/790455011.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
/Users/yossefdozli/Projects/vector_db_live_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13225.57it/s]


Document(metadata={}, page_content='REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL\nof 13 June 2024\nlaying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008,\n(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and\nDirectives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)\n(Text with EEA relevance)\nTHE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,\nHaving regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,\nHaving regard to the proposal from the European Commission,\nAfter transmission of the draft legislative act to the national parliaments,\nHaving regard to the opinion of the European Economic and Social Committee (1),\nHaving regard to the opinion of the European Central Bank (2),\nHaving regard to the opinion of the Committee of the Regions (3),\nActing in accordance with the ordinar

In [10]:
start_index = 0
chunks_with_metadata = []
for idx, chunk in enumerate(chunks):
    doc_chunk = DocumentChunk(
                id= "eu_ai_act_{}".format(idx),
                source = "eu_ai_act",
                chunk_index = idx,
                content = chunk.page_content,
                metadata = ChunkMetadata(
                    chunking_method = "semantic",
                    source_info = "eu_ai_act",
                    extra_details = {
                        "total_chunks": len(chunks),
                        "start_index" : start_index
                                    }
                )
    )

    chunks_with_metadata.append(
        doc_chunk
    )
    start_index += doc_chunk.length
    

In [11]:
print(len(chunks_with_metadata))
print(f"{sum(doc_chunk.length for doc_chunk in chunks_with_metadata)/len(chunks_with_metadata):.0f}")

129
4637


In [12]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
vectors = model.encode(inputs=[c.content for c in chunks_with_metadata],show_progress_bar=True,normalize_embeddings=True, convert_to_numpy=True)
print(vectors[0])
print(vectors.shape)
print(len(chunks_with_metadata) == len(vectors))

Batches: 100%|██████████| 5/5 [00:00<00:00,  5.03it/s]

[-2.88874246e-02  2.08210945e-02 -1.05953058e-02 -7.01444969e-02
  4.44227271e-02  5.17414212e-02 -4.61174268e-03  6.54940456e-02
 -1.51081048e-02 -4.01491113e-03 -1.94936115e-02 -3.28062735e-02
 -3.75169404e-02  2.69647352e-02 -2.24052519e-02 -5.34243742e-03
 -2.83758482e-03 -4.36321758e-02 -7.53594041e-02  2.86694411e-02
  5.92440702e-02 -3.02103851e-02 -6.93334714e-02  9.13930964e-03
 -1.13506094e-01 -1.00724418e-02 -2.85723731e-02 -1.52725965e-01
 -2.86402982e-02  1.28452387e-02 -2.12283600e-02 -2.25045197e-02
 -1.89492607e-03 -8.59370176e-03  1.12490300e-02 -9.52703208e-02
  2.54679546e-02 -9.32463109e-02  1.72486727e-03  9.13968496e-03
 -3.64467711e-03 -7.86033794e-02 -1.32136613e-01 -1.79686947e-04
  4.19944674e-02  1.50891826e-01  3.37562114e-02 -5.38553521e-02
 -1.63892925e-01  3.70795233e-03 -7.06137717e-02  6.76139630e-03
 -8.20730929e-04 -3.37652303e-03  6.01820555e-03 -5.59297651e-02
 -4.85164486e-02  7.42917601e-03  7.04570115e-02  1.06955739e-02
  7.87902176e-02 -8.48989

In [13]:
from qdrant_client import QdrantClient, models

qdrant_client = QdrantClient(url="http://localhost:6333")
exists = qdrant_client.collection_exists(collection_name="eu")
if not exists:
    qdrant_client.create_collection(
        collection_name="eu",
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
    )
print(qdrant_client.get_collections())

collections=[CollectionDescription(name='eu')]


In [14]:
# from bertopic import BERTopic
# from umap import UMAP
# umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
# topic_model = BERTopic(umap_model=umap_model, min_topic_size=5)

# topic_label, _ = topic_model.fit_transform([chunk.content for chunk in chunks_with_metadata], embeddings=vectors)
# topic_model.get_topic(1)

In [15]:

from datetime import datetime


timestamp = datetime.now().isoformat()
payloads =  [{
            "source_filename": chunk.source,
            "original_text": chunk.content,
            "timestamp": timestamp,
            "tags": [ "EU_AI_Act", "Legal_Document"],
            "chunk_index": chunk.chunk_index,
            "chunk_length": chunk.length,
            "metadata": chunk.metadata
        } for chunk in chunks_with_metadata]


qdrant_client.upsert (
    collection_name="eu",
    points=models.Batch(
        ids=[c.chunk_index for c in chunks_with_metadata],
        vectors= vectors.tolist(),
        payloads=[c.model_dump() for c in chunks_with_metadata]
    )
)


UpdateResult(operation_id=5, status=<UpdateStatus.COMPLETED: 'completed'>)

In [16]:

info =qdrant_client.get_collection(collection_name="eu")
print(info.config.params.vectors.size)      # 384
print(info.config.params.vectors.distance)  # Cosine
print(info.points_count) 
print(info.points_count == len(chunks_with_metadata))

384
Cosine
129
True


In [17]:
test_points = qdrant_client.scroll(
    collection_name='eu',
    limit=3,
    with_payload=True,
    with_vectors=False
)

for point in test_points:
    print(point)


[Record(id=0, payload={'id': 'eu_ai_act_0', 'source': 'eu_ai_act', 'chunk_index': 0, 'content': 'REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL\nof 13 June 2024\nlaying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008,\n(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and\nDirectives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)\n(Text with EEA relevance)\nTHE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,\nHaving regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,\nHaving regard to the proposal from the European Commission,\nAfter transmission of the draft legislative act to the national parliaments,\nHaving regard to the opinion of the European Economic and Social Committee (1),\nHaving regard to the opinion of the European Central Bank (2),\nHaving regard to the opinion of the Committ

In [19]:
# question = input("Ask a question about the EU AI Act: ")
sample_question = "What are the requirements for high-risk AI systems"
question_vector = model.encode([sample_question],normalize_embeddings=True).tolist()
answers = qdrant_client.query_points(
    collection_name='eu',
    query=question_vector[0],
    limit=5,
    with_payload=True,
    with_vectors=False
)


In [ ]:
for answer in answers.points:
      print(f"score={answer.score:.3f} | topic={answer.payload}")
      print("-" * 60)

score=0.694 | topic={'id': 'eu_ai_act_40', 'source': 'eu_ai_act', 'chunk_index': 40, 'content': 'High-risk AI systems shall be designed and developed in such a way, including with appropriate human-machine\ninterface tools, that they can be effectively overseen by natural persons during the period in which they are in use. 2. Human oversight shall aim to prevent or minimise the risks to health, safety or fundamental rights that may emerge\nwhen a high-risk AI system is used in accordance with its intended purpose or under conditions of reasonably foreseeable\nmisuse, in particular where such risks persist despite the application of other requirements set out in this Section. 3. The oversight measures shall be commensurate with the risks, level of autonomy and context of use of the high-risk\nAI system, and shall be ensured through either one or both of the following types of measures:\n(a) measures identified and built, when technically feasible, into the high-risk AI system by the pro

### Milestone 5 ###

In [20]:
from qdrant_client.http.models import QueryResponse

test1 = "What are the requirements for high-risk AI systems?"
test2 = "What is the definition of high-risk AI systems?"
test3 = "who is obligated to adhere to the EU AI Act?"
test4 = "What is the purpose of the EU AI Act?"

test_questions = [test1, test2, test3, test4]

def query_eu_ai_act(question) -> QueryResponse:
    question_vector = model.encode([question], normalize_embeddings=True).tolist()
    answers = qdrant_client.query_points(
        collection_name='eu',
        query=question_vector[0],
        limit=5,
        with_payload=True,
        with_vectors=False
    )
    return answers

answers = {
      i: query_eu_ai_act(q).points
      for i, q in enumerate(test_questions, start=1)
  }

In [21]:
avg_score = {
    i: sum(answer.score for answer in answers[i]) / len(answers[i])
    for i in answers
}
max_score = {
    i: max(answer.score for answer in answers[i]) 
    for i in answers
}
min_score = {
    i: min(answer.score for answer in answers[i]) 
    for i in answers
}
print("avg_score", avg_score)
print("max_score", max_score)
print("min_score", min_score)

flat = [hit   for hits in answers.values()   for hit in hits]
avg_total = sum(hit.score for hit in flat) / len(flat) if flat else 0
print("avg_total", avg_total)



avg_score {1: 0.61513609, 2: 0.615786712, 3: 0.67107641, 4: 0.614339674}
max_score {1: 0.69384265, 2: 0.6838451, 3: 0.68803763, 4: 0.6330283}
min_score {1: 0.5813712, 2: 0.57367545, 3: 0.6586595, 4: 0.59734213}
avg_total 0.6290847215


In [60]:
best 
import textwrap

for  n, h in answers.items():
      print("=" * 100)
      print(f"QUERY {n}: {test_questions[n - 1]}")
      print("-" * 100)
      for i in h:
        print(f"score={i.score:.3f}")
        print("-" * 100)
        print(textwrap.fill(i.payload["content"].replace("\n", " "), width=100))
        print("-" * 100)
      print()


QUERY 1: What are the requirements for high-risk AI systems?
----------------------------------------------------------------------------------------------------
score=0.694
----------------------------------------------------------------------------------------------------
High-risk AI systems shall be designed and developed in such a way, including with appropriate
human-machine interface tools, that they can be effectively overseen by natural persons during the
period in which they are in use. 2. Human oversight shall aim to prevent or minimise the risks to
health, safety or fundamental rights that may emerge when a high-risk AI system is used in
accordance with its intended purpose or under conditions of reasonably foreseeable misuse, in
particular where such risks persist despite the application of other requirements set out in this
Section. 3. The oversight measures shall be commensurate with the risks, level of autonomy and
context of use of the high-risk AI system, and shall be